# GUI vs CLI Head-to-Head (TLG1025, HPC)

This notebook compares MS-DIAL GUI (`msdial-brian`) versus MS-DIAL CLI runs at both feature-level and group-behavior levels.

## Goals
- Build one-to-one feature matches using mz/RT tolerances.
- Quantify agreement of feature behavior across biological sample groups.
- Zoom in on non-matching and behavior-discordant features.
- Export drilldown tables for manual review.

Adapted from `metabolomics-sandbox/benchmarking/analysis/02_gui_vs_cli_v3_head_to_head.ipynb` for HPC paths.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

# --- Paths ---
SANDBOX = Path("/hpc/mydata/anthony.goering/repos/metabolomics-sandbox")
RESULTS_DIR = SANDBOX / "benchmarking" / "TLG1025" / "results"
HPC_DIR = Path("/hpc/projects/mass_spec_chi/Team/Tony/msdial-tlg1025")

GUI_PATH = RESULTS_DIR / "msdial-brian" / "msdial-brian_feature_matrix.csv"
CLI_PATH = HPC_DIR / "msdial-console-v4-blankon-refmatchoff_feature_matrix.csv"

OUTPUT_DIR = Path("/hpc/mydata/anthony.goering/repos/Rapid-QC-MS/prototypes/msdial-console/notebooks/output/gui-vs-cli-v4-head-to-head")

MATCH_PPM = 10.0
MATCH_RT = 0.15
RELAXED_PPM = 20.0
RELAXED_RT = 0.30
DISCORDANT_PEARSON_MAX = 0.50

POLARITIES = ["pos", "neg"]

print(f"GUI matrix: {GUI_PATH}")
print(f"CLI matrix: {CLI_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

GUI matrix: /hpc/mydata/anthony.goering/repos/metabolomics-sandbox/benchmarking/TLG1025/results/msdial-brian/msdial-brian_feature_matrix.csv
CLI matrix: /hpc/projects/mass_spec_chi/Team/Tony/msdial-tlg1025/msdial-console-v4-blankon-refmatchoff_feature_matrix.csv
Output dir: /hpc/mydata/anthony.goering/repos/Rapid-QC-MS/prototypes/msdial-console/notebooks/output/gui-vs-cli-v4-head-to-head


In [2]:
def _sample_type_and_class(sample_name: str) -> tuple[str, str]:
    if sample_name.startswith("BK_"):
        return "Blank", "blank"
    if sample_name.startswith("QC_"):
        return "QC", "pool"
    token = sample_name.split("_", 1)[0]
    token = re.sub(r"^uninfected", "uninf", token)
    return "Sample", token


def _sample_polarity(sample_name: str) -> str:
    if "_Pos_" in sample_name:
        return "pos"
    if "_Neg_" in sample_name:
        return "neg"
    return "unknown"


def _class_sort_key(class_id: str) -> tuple[str, int]:
    m = re.match(r"^([a-zA-Z]+)(\d+)hr$", class_id)
    if m:
        return (m.group(1), int(m.group(2)))
    if class_id == "pool":
        return ("zz_pool", -1)
    if class_id == "blank":
        return ("zz_blank", -1)
    return (class_id, -1)


def extract_sample_metadata(df: pd.DataFrame) -> pd.DataFrame:
    intensity_cols = [c for c in df.columns if c.startswith("intensity_")]
    records = []
    for col in intensity_cols:
        sample_name = col.replace("intensity_", "", 1)
        sample_type, class_id = _sample_type_and_class(sample_name)
        records.append(
            {
                "column": col,
                "sample_name": sample_name,
                "sample_type": sample_type,
                "class_id": class_id,
                "polarity": _sample_polarity(sample_name),
            }
        )

    meta = pd.DataFrame(records)
    if meta.empty:
        return meta

    class_order = {cls: i for i, cls in enumerate(sorted(meta["class_id"].unique(), key=_class_sort_key))}
    meta["class_order"] = meta["class_id"].map(class_order)
    meta = meta.sort_values(
        by=["polarity", "sample_type", "class_order", "sample_name"]
    ).drop(columns=["class_order"]).reset_index(drop=True)

    return meta


def load_feature_matrix(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    required = {"feature_id", "mz", "rt", "polarity"}
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {path.name}: {missing}")

    df["feature_id"] = df["feature_id"].astype(str)
    df["mz"] = pd.to_numeric(df["mz"], errors="coerce").fillna(0.0)
    df["rt"] = pd.to_numeric(df["rt"], errors="coerce").fillna(0.0)
    df["polarity"] = df["polarity"].astype(str).str.lower()

    for col in [c for c in df.columns if c.startswith("intensity_")]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df

In [3]:
def match_features_one_to_one(
    gui_df: pd.DataFrame,
    cli_df: pd.DataFrame,
    ppm_tol: float,
    rt_tol: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    gui = gui_df.reset_index(drop=True).copy()
    cli = cli_df.reset_index(drop=True).copy()

    gui_mz = gui["mz"].to_numpy(dtype=float)
    gui_rt = gui["rt"].to_numpy(dtype=float)

    cli_mz_raw = cli["mz"].to_numpy(dtype=float)
    cli_rt_raw = cli["rt"].to_numpy(dtype=float)
    cli_order = np.argsort(cli_rt_raw)
    cli_rt = cli_rt_raw[cli_order]
    cli_mz = cli_mz_raw[cli_order]

    candidates: list[tuple[float, float, float, int, int]] = []
    for gi, (mz, rt) in enumerate(zip(gui_mz, gui_rt)):
        left = np.searchsorted(cli_rt, rt - rt_tol, side="left")
        right = np.searchsorted(cli_rt, rt + rt_tol, side="right")
        if right <= left:
            continue

        local_idx = np.arange(left, right)
        ppm_err = np.abs(cli_mz[local_idx] - mz) / max(abs(mz), 1e-12) * 1e6
        keep = np.where(ppm_err <= ppm_tol)[0]
        if keep.size == 0:
            continue

        for k in keep:
            li = int(local_idx[k])
            ci = int(cli_order[li])
            rt_err = float(cli_rt[li] - rt)
            ppm = float(ppm_err[k])
            score = abs(rt_err) / rt_tol + ppm / ppm_tol
            candidates.append((score, abs(rt_err), ppm, gi, ci))

    candidates.sort(key=lambda x: (x[0], x[1], x[2]))

    used_gui: set[int] = set()
    used_cli: set[int] = set()
    rows = []
    for score, abs_rt_err, ppm_err, gi, ci in candidates:
        if gi in used_gui or ci in used_cli:
            continue
        used_gui.add(gi)
        used_cli.add(ci)

        g = gui.iloc[gi]
        c = cli.iloc[ci]
        rows.append(
            {
                "gui_feature_id": g["feature_id"],
                "cli_feature_id": c["feature_id"],
                "gui_mz": float(g["mz"]),
                "gui_rt": float(g["rt"]),
                "cli_mz": float(c["mz"]),
                "cli_rt": float(c["rt"]),
                "ppm_error": ppm_err,
                "rt_error_abs": abs_rt_err,
                "match_score": score,
            }
        )

    matched = pd.DataFrame(rows).sort_values("match_score").reset_index(drop=True)

    gui_only = gui.loc[[i for i in range(len(gui)) if i not in used_gui]].reset_index(drop=True)
    cli_only = cli.loc[[i for i in range(len(cli)) if i not in used_cli]].reset_index(drop=True)

    return matched, gui_only, cli_only


def build_group_profiles(
    df_pol: pd.DataFrame,
    sample_meta: pd.DataFrame,
    polarity: str,
) -> tuple[pd.DataFrame, list[str]]:
    sample_meta_pol = sample_meta[
        (sample_meta["polarity"] == polarity) & (sample_meta["sample_type"] == "Sample")
    ].copy()
    classes = sorted(sample_meta_pol["class_id"].unique(), key=_class_sort_key)

    df_idx = df_pol.set_index("feature_id")
    profiles = pd.DataFrame(index=df_idx.index)
    for class_id in classes:
        cols = sample_meta_pol.loc[sample_meta_pol["class_id"] == class_id, "column"].tolist()
        if not cols:
            continue
        profiles[class_id] = df_idx[cols].mean(axis=1)

    return profiles, list(profiles.columns)


def _pearson_log_safe(a: np.ndarray, b: np.ndarray) -> float:
    x = np.log1p(a)
    y = np.log1p(b)
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def _cosine_safe(a: np.ndarray, b: np.ndarray) -> float:
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na == 0.0 or nb == 0.0:
        return np.nan
    return float(np.dot(a, b) / (na * nb))


def characterize_behavior(
    matched_df: pd.DataFrame,
    gui_profiles: pd.DataFrame,
    cli_profiles: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    rows = []
    for row in matched_df.itertuples(index=False):
        gui_vec = gui_profiles.loc[row.gui_feature_id, group_cols].to_numpy(dtype=float)
        cli_vec = cli_profiles.loc[row.cli_feature_id, group_cols].to_numpy(dtype=float)

        pearson_log = _pearson_log_safe(gui_vec, cli_vec)
        cosine = _cosine_safe(gui_vec, cli_vec)
        rmse_log2 = float(np.sqrt(np.mean((np.log2(gui_vec + 1.0) - np.log2(cli_vec + 1.0)) ** 2)))

        log2_diffs = np.abs(np.log2((gui_vec + 1.0) / (cli_vec + 1.0)))
        max_idx = int(np.argmax(log2_diffs))
        all_zero_profile = bool(np.all(gui_vec == 0.0) and np.all(cli_vec == 0.0))

        rows.append(
            {
                "gui_feature_id": row.gui_feature_id,
                "cli_feature_id": row.cli_feature_id,
                "pearson_log_group_profile": pearson_log,
                "cosine_group_profile": cosine,
                "rmse_log2_group_profile": rmse_log2,
                "max_group": group_cols[max_idx],
                "max_group_log2_diff": float(log2_diffs[max_idx]),
                "max_group_gui_mean": float(gui_vec[max_idx]),
                "max_group_cli_mean": float(cli_vec[max_idx]),
                "all_zero_profile": all_zero_profile,
            }
        )

    return pd.DataFrame(rows)


def best_relaxed_candidate(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    ppm_tol: float,
    rt_tol: float,
    target_prefix: str,
) -> pd.DataFrame:
    target = target_df.reset_index(drop=True)
    t_mz_raw = target["mz"].to_numpy(dtype=float)
    t_rt_raw = target["rt"].to_numpy(dtype=float)
    order = np.argsort(t_rt_raw)
    t_mz = t_mz_raw[order]
    t_rt = t_rt_raw[order]

    records = []
    for row in source_df.itertuples(index=False):
        mz = float(row.mz)
        rt = float(row.rt)

        left = np.searchsorted(t_rt, rt - rt_tol, side="left")
        right = np.searchsorted(t_rt, rt + rt_tol, side="right")

        if right <= left:
            records.append(
                {
                    "source_feature_id": row.feature_id,
                    f"nearest_{target_prefix}_feature_id": None,
                    "nearest_ppm_error": np.nan,
                    "nearest_rt_error_abs": np.nan,
                }
            )
            continue

        local_idx = np.arange(left, right)
        ppm_err = np.abs(t_mz[local_idx] - mz) / max(abs(mz), 1e-12) * 1e6
        score = np.abs(t_rt[local_idx] - rt) / rt_tol + ppm_err / ppm_tol
        best_k = int(np.argmin(score))

        best_local = int(local_idx[best_k])
        best_target_idx = int(order[best_local])
        best_row = target.iloc[best_target_idx]

        records.append(
            {
                "source_feature_id": row.feature_id,
                f"nearest_{target_prefix}_feature_id": best_row["feature_id"],
                "nearest_ppm_error": float(ppm_err[best_k]),
                "nearest_rt_error_abs": float(abs(t_rt[best_local] - rt)),
            }
        )

    return pd.DataFrame(records)

In [4]:
gui = load_feature_matrix(GUI_PATH)
cli = load_feature_matrix(CLI_PATH)

gui_meta = extract_sample_metadata(gui)
cli_meta = extract_sample_metadata(cli)

print(f"GUI rows: {len(gui):,}")
print(f"CLI rows: {len(cli):,}")

gui_cols = set(gui_meta["column"])
cli_cols = set(cli_meta["column"])

print(f"Shared intensity columns: {len(gui_cols & cli_cols)}")
print(f"Missing in CLI: {len(gui_cols - cli_cols)}")
print(f"Missing in GUI: {len(cli_cols - gui_cols)}")

GUI rows: 14,056
CLI rows: 14,626
Shared intensity columns: 232
Missing in CLI: 0
Missing in GUI: 0


In [5]:
match_results: dict[str, dict[str, pd.DataFrame]] = {}
match_summary_rows: list[dict[str, object]] = []

base_cols = ["feature_id", "mz", "rt", "adduct", "occurrence_overall"]

for pol in POLARITIES:
    gui_pol = gui.loc[gui["polarity"] == pol, base_cols].copy()
    cli_pol = cli.loc[cli["polarity"] == pol, base_cols].copy()

    matched, gui_only, cli_only = match_features_one_to_one(
        gui_pol, cli_pol, ppm_tol=MATCH_PPM, rt_tol=MATCH_RT,
    )

    matched["polarity"] = pol
    gui_only["polarity"] = pol
    cli_only["polarity"] = pol

    match_results[pol] = {"matched": matched, "gui_only": gui_only, "cli_only": cli_only}

    match_summary_rows.append({
        "polarity": pol,
        "gui_features": len(gui_pol),
        "cli_features": len(cli_pol),
        "matched_pairs": len(matched),
        "gui_match_rate": round(len(matched) / len(gui_pol) * 100, 2) if len(gui_pol) else np.nan,
        "cli_match_rate": round(len(matched) / len(cli_pol) * 100, 2) if len(cli_pol) else np.nan,
        "gui_only": len(gui_only),
        "cli_only": len(cli_only),
    })

match_summary = pd.DataFrame(match_summary_rows)
print("Feature matching summary:")
match_summary

Feature matching summary:


,polarity,gui_features,cli_features,matched_pairs,gui_match_rate,cli_match_rate,gui_only,cli_only
0,pos,7823,8224,7258,92.78,88.25,565,966
1,neg,6233,6402,6083,97.59,95.02,150,319


In [6]:
behavior_frames: list[pd.DataFrame] = []
behavior_summary_rows: list[dict[str, object]] = []

for pol in POLARITIES:
    matched = match_results[pol]["matched"].copy()
    gui_pol_full = gui.loc[gui["polarity"] == pol].copy()
    cli_pol_full = cli.loc[cli["polarity"] == pol].copy()

    gui_profiles, gui_groups = build_group_profiles(gui_pol_full, gui_meta, pol)
    cli_profiles, cli_groups = build_group_profiles(cli_pol_full, cli_meta, pol)

    shared_groups = [g for g in gui_groups if g in cli_groups]

    behavior = characterize_behavior(matched, gui_profiles, cli_profiles, shared_groups)
    behavior["polarity"] = pol
    behavior_frames.append(behavior)

    behavior_summary_rows.append({
        "polarity": pol,
        "matched_pairs": len(behavior),
        "median_pearson_log_profile": round(behavior["pearson_log_group_profile"].median(), 4),
        "p25_pearson_log_profile": round(behavior["pearson_log_group_profile"].quantile(0.25), 4),
        "fraction_pearson_ge_0.9": round((behavior["pearson_log_group_profile"] >= 0.9).mean(), 4),
        "fraction_pearson_ge_0.75": round((behavior["pearson_log_group_profile"] >= 0.75).mean(), 4),
        "median_cosine_profile": round(behavior["cosine_group_profile"].median(), 4),
        "median_rmse_log2_profile": round(behavior["rmse_log2_group_profile"].median(), 6),
    })

behavior_df = pd.concat(behavior_frames, ignore_index=True)
behavior_summary = pd.DataFrame(behavior_summary_rows)

print("Feature-level behavior agreement summary:")
behavior_summary

Feature-level behavior agreement summary:


,polarity,matched_pairs,median_pearson_log_profile,p25_pearson_log_profile,fraction_pearson_ge_0.9,fraction_pearson_ge_0.75,median_cosine_profile,median_rmse_log2_profile
0,pos,7258,1.0,0.9963,0.9107,0.9574,1.0,0.000006
1,neg,6083,1.0,1.0000,0.9786,0.9905,1.0,0.000003


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BG = "#0b0b0b"
AX_BG = "#121216"
FG = "#f5f5f5"
GRID = "#38384a"
POL_COLOR = {"pos": "#7c6cff", "neg": "#b39dff"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, facecolor=BG)

for ax, pol in zip(axes, ["pos", "neg"]):
    vals = behavior_df.loc[
        behavior_df["polarity"] == pol, "pearson_log_group_profile",
    ].dropna().to_numpy()

    ax.set_facecolor(AX_BG)
    ax.hist(vals, bins=40, range=(-1, 1), color=POL_COLOR[pol], alpha=0.9,
            edgecolor="#d8d2ff", linewidth=0.3)
    ax.axvline(0.75, linestyle="--", linewidth=1.3, color="#ffb86b", label="0.75")
    ax.axvline(0.90, linestyle="--", linewidth=1.3, color="#6ee7b7", label="0.90")
    if len(vals):
        ax.axvline(float(np.quantile(vals, 0.25)), linestyle=":", linewidth=1.6,
                    color="#f78cda", label="p25")
    ax.set_title(f"{pol.upper()} Pearson(log profile)", color=FG, fontsize=12, pad=8)
    ax.set_xlabel("pearson_log_group_profile", color=FG)
    ax.set_xlim(-1, 1)
    ax.grid(alpha=0.35, color=GRID, linewidth=0.8)
    ax.tick_params(colors=FG)
    for spine in ax.spines.values():
        spine.set_color("#5a5a6a")

axes[0].set_ylabel("Matched feature count", color=FG)
legend = axes[1].legend(loc="upper left", frameon=False, fontsize=10)
for text in legend.get_texts():
    text.set_color(FG)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "pearson_distribution.png"), dpi=150, facecolor=BG, bbox_inches="tight")
plt.show()
print("Saved pearson_distribution.png")

Saved pearson_distribution.png


In [8]:
# Unmatched feature analysis
gui_only_frames = []
cli_only_frames = []

for pol in POLARITIES:
    gui_only = match_results[pol]["gui_only"].copy()
    cli_only = match_results[pol]["cli_only"].copy()

    cli_pol = cli.loc[cli["polarity"] == pol, ["feature_id", "mz", "rt"]].copy()
    gui_pol = gui.loc[gui["polarity"] == pol, ["feature_id", "mz", "rt"]].copy()

    gui_relaxed = best_relaxed_candidate(gui_only[["feature_id", "mz", "rt"]], cli_pol, RELAXED_PPM, RELAXED_RT, "cli")
    gui_only = gui_only.merge(gui_relaxed, left_on="feature_id", right_on="source_feature_id", how="left").drop(columns=["source_feature_id"])

    cli_relaxed = best_relaxed_candidate(cli_only[["feature_id", "mz", "rt"]], gui_pol, RELAXED_PPM, RELAXED_RT, "gui")
    cli_only = cli_only.merge(cli_relaxed, left_on="feature_id", right_on="source_feature_id", how="left").drop(columns=["source_feature_id"])

    gui_only["has_relaxed_neighbor"] = (
        gui_only["nearest_cli_feature_id"].notna()
        & (gui_only["nearest_ppm_error"] <= RELAXED_PPM)
        & (gui_only["nearest_rt_error_abs"] <= RELAXED_RT)
    )
    cli_only["has_relaxed_neighbor"] = (
        cli_only["nearest_gui_feature_id"].notna()
        & (cli_only["nearest_ppm_error"] <= RELAXED_PPM)
        & (cli_only["nearest_rt_error_abs"] <= RELAXED_RT)
    )

    gui_only_frames.append(gui_only)
    cli_only_frames.append(cli_only)

gui_only_all = pd.concat(gui_only_frames, ignore_index=True)
cli_only_all = pd.concat(cli_only_frames, ignore_index=True)

unmatched_summary = pd.DataFrame([
    {
        "set": "gui_only", "n_features": len(gui_only_all),
        "fraction_with_relaxed_neighbor": round(gui_only_all["has_relaxed_neighbor"].mean() * 100, 2),
        "median_relaxed_ppm_error": gui_only_all["nearest_ppm_error"].median(),
        "median_relaxed_rt_error": gui_only_all["nearest_rt_error_abs"].median(),
    },
    {
        "set": "cli_only", "n_features": len(cli_only_all),
        "fraction_with_relaxed_neighbor": round(cli_only_all["has_relaxed_neighbor"].mean() * 100, 2),
        "median_relaxed_ppm_error": cli_only_all["nearest_ppm_error"].median(),
        "median_relaxed_rt_error": cli_only_all["nearest_rt_error_abs"].median(),
    },
])

print("Unmatched features with relaxed-tolerance neighbors:")
unmatched_summary

Unmatched features with relaxed-tolerance neighbors:


,set,n_features,fraction_with_relaxed_neighbor,median_relaxed_ppm_error,median_relaxed_rt_error
0,gui_only,715,30.07,46.768608,0.08
1,cli_only,1285,45.68,29.350199,0.08


In [9]:
# Export results
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

matched_all = pd.concat([match_results[p]["matched"] for p in POLARITIES], ignore_index=True)
matched_with_behavior = matched_all.merge(behavior_df, on=["polarity", "gui_feature_id", "cli_feature_id"], how="left")

discordant_matched = matched_with_behavior[
    (~matched_with_behavior["all_zero_profile"])
    & (matched_with_behavior["pearson_log_group_profile"].fillna(-1.0) < DISCORDANT_PEARSON_MAX)
].sort_values(["polarity", "pearson_log_group_profile", "max_group_log2_diff"],
              ascending=[True, True, False]).reset_index(drop=True)

match_summary.to_csv(OUTPUT_DIR / "match_summary.csv", index=False)
matched_all.to_csv(OUTPUT_DIR / "matched_pairs.csv", index=False)
behavior_summary.to_csv(OUTPUT_DIR / "behavior_summary.csv", index=False)
behavior_df.to_csv(OUTPUT_DIR / "behavior_per_feature.csv", index=False)
gui_only_all.to_csv(OUTPUT_DIR / "gui_only_features.csv", index=False)
cli_only_all.to_csv(OUTPUT_DIR / "cli_only_features.csv", index=False)
unmatched_summary.to_csv(OUTPUT_DIR / "unmatched_summary.csv", index=False)
discordant_matched.to_csv(OUTPUT_DIR / "discordant_matched_features.csv", index=False)

print(f"Wrote outputs to: {OUTPUT_DIR}")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  - {p.name}")

print(f"\nDiscordant matched features (pearson < {DISCORDANT_PEARSON_MAX}): {len(discordant_matched)}")
discordant_matched.head(20)

Wrote outputs to: /hpc/mydata/anthony.goering/repos/Rapid-QC-MS/prototypes/msdial-console/notebooks/output/gui-vs-cli-v4-head-to-head
  - behavior_per_feature.csv
  - behavior_summary.csv
  - cli_only_features.csv
  - discordant_matched_features.csv
  - gui_only_features.csv
  - match_summary.csv
  - matched_pairs.csv
  - unmatched_summary.csv

Discordant matched features (pearson < 0.5): 135


,gui_feature_id,cli_feature_id,gui_mz,gui_rt,cli_mz,cli_rt,ppm_error,rt_error_abs,match_score,polarity,pearson_log_group_profile,cosine_group_profile,rmse_log2_group_profile,max_group,max_group_log2_diff,max_group_gui_mean,max_group_cli_mean,all_zero_profile
0,msdial_brian_neg_003596,msdial_console_v4_blankon_refmatchoff_neg_003671,259.8159,6.98,259.8158,6.85,0.384888,0.13,0.905155,neg,-0.876055,0.695483,1.825314,uninf3hr,4.466315,1.649937e+07,7.464010e+05,False
1,msdial_brian_neg_003600,msdial_console_v4_blankon_refmatchoff_neg_003676,259.8216,6.88,259.8216,6.96,0.000000,0.08,0.533333,neg,-0.148063,0.884518,0.969209,zikv9hr,2.570692,1.981196e+06,1.177017e+07,False
2,msdial_brian_neg_000732,msdial_console_v4_blankon_refmatchoff_neg_000758,128.0344,7.56,128.0344,7.58,0.000000,0.02,0.133333,neg,-0.147872,0.959513,0.431553,uninf3hr,1.646835,1.341360e+06,4.283493e+05,False
3,msdial_brian_neg_004280,msdial_console_v4_blankon_refmatchoff_neg_004377,302.8013,1.27,302.8013,1.33,0.000000,0.06,0.400000,neg,0.037065,0.890266,1.970471,zikv18hr,3.290112,2.159267e+04,2.112256e+05,False
4,msdial_brian_neg_000064,msdial_console_v4_blankon_refmatchoff_neg_000063,68.9948,4.40,68.9948,4.52,0.000000,0.12,0.800000,neg,0.057470,0.990689,0.187355,denv36hr,0.494396,6.478873e+05,9.126993e+05,False
5,msdial_brian_neg_005503,msdial_console_v4_blankon_refmatchoff_neg_005635,439.6833,6.74,439.6833,6.73,0.000000,0.01,0.066667,neg,0.313875,0.480414,12.107153,denv6hr,18.400293,0.000000e+00,3.459702e+05,False
6,msdial_brian_neg_002525,msdial_console_v4_blankon_refmatchoff_neg_002583,207.8360,0.82,207.8363,0.73,1.443446,0.09,0.744345,neg,0.332098,0.828405,1.945208,denv18hr,2.987956,1.026863e+05,8.146679e+05,False
7,msdial_brian_neg_002249,msdial_console_v4_blankon_refmatchoff_neg_002289,195.8273,6.65,195.8272,6.65,0.510654,0.00,0.051065,neg,0.398919,0.977391,0.445698,denv33hr,1.323794,1.387894e+06,5.544393e+05,False
8,msdial_brian_neg_002684,msdial_console_v4_blankon_refmatchoff_neg_002742,215.0323,7.46,215.0323,7.46,0.000000,0.00,0.000000,neg,0.420813,0.945707,0.689785,denv6hr,3.573682,2.127645e+06,2.533290e+07,False
9,msdial_brian_neg_001554,msdial_console_v4_blankon_refmatchoff_neg_001588,168.0509,8.81,168.0508,8.86,0.595058,0.05,0.392839,neg,0.449494,0.564115,1.809950,denv33hr,4.190630,3.284237e+05,5.997098e+06,False
